 Анализ генома тихоходки Ramazzottius varieornatus

Цель проекта: выявить белки, ассоциированные с ДНК у экстремофильной тихоходки R. varieornatus, которые могут объяснить её устойчивость к радиации.  



Шаг №1: подготовка и загрузка данных

In [ ]:
mkdir -p ~/tardigrade_project/{data,blast_db,peptides,results}
cd ~/tardigrade_project

# установка инструментов через conda
conda install -c bioconda blast samtools seqtk diamond

cd data
wget ftp.ncbi.nlm.nih.gov/genomes/all/GCA/001/949/185/GCA_001949185.1_Rvar_4.0/GCA_001949185.1_Rvar_4.0_genomic.fna.gz
gunzip GCA_001949185.1_Rvar_4.0_genomic.fna.gz

Шаг №2: анализ белков из полученных данных из готового файла с белками, потому что мощности не хватило

In [ ]:

cp ~/Downloads/peptides.txt ~/tardigrade_project/peptides/
cd ~/tardigrade_project/data
grep -c "^>" Rvar_proteins.faa # считаем количество предсказанных белков

Шаг №3: поиск пептидов в базе  blastp

In [ ]:
cd ~/tardigrade_project/blast_db
makeblastdb -in ../data/Rvar_proteins.faa -dbtype prot -out Rvar_proteins # соездаем базу из белков тихоходки

In [ ]:
cd ~/tardigrade_project
blastp -db blast_db/Rvar_proteins -query peptides/peptides.fasta -out results/peptide_matches.txt -outfmt "6 qseqid sseqid evalue pident length qcovs stitle" -max_target_seqs 1 # ищем
cut -f2 results/peptide_matches.txt | sort -u > results/candidate_protein_ids.txt # достаем уникальные идентификаторы белков

In [ ]:
cd ~/tardigrade_project/data
samtools faidx Rvar_proteins.faa # достаем последовательность потенциально интересных белков

In [ ]:
xargs -a ../results/candidate_protein_ids.txt samtools faidx Rvar_proteins.faa > ../results/candidate_proteins.faa # делаем файлик с сопоставление последовательностей и названий

Шаг №4: смотрим локализацию

использую WoLF PSORT, дальше выбрать Organism type: Eukaryota, загрузить файл candidate_proteins.faa

Шаг №5: поиск гомологи

In [ ]:
cd ~/tardigrade_project
blastp -db uniprotkb_swissprot -query results/candidate_proteins.faa -out results/swissprot_blast.txt -outfmt "6 qseqid sseqid evalue pident qcovs stitle" -max_target_seqs 1

Шаг №6: ищем консервативные домены с помощью сервиса HMMER\

Database: Pfam, последовательности из файла candidate_proteins.faa и поиск

Шаг №7: ура, пишем отчет...